# 1Day Start 役職推定パイプライン

2day版をベースにした 1day 開始用ノートブックです。

- day_filter は 0 を使用
- day2 制約（処刑/襲撃制約）は無効
- ログ表示は表形式に寄せて見やすく整理

In [1]:
import sys
import json
import time
import traceback
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, classification_report, confusion_matrix

import xgboost as xgb
import optuna
import joblib

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_PATH = PROJECT_ROOT / "src"
CONFIG_PATH = PROJECT_ROOT / "config" / "training_config.json"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from src.pipelines.training_pipeline import load_training_config, get_data_paths

print(pd.DataFrame([
    {"key": "PROJECT_ROOT", "value": str(PROJECT_ROOT)},
    {"key": "CONFIG_PATH", "value": str(CONFIG_PATH)},
    {"key": "python", "value": sys.version.split()[0]},
]))

C:\Users\takic\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ModuleNotFoundError: No module named 'src'

In [32]:
RUN_OPTIONS = {
    "day_filter": 0,
    "day2_flag": False,
    "n_trials": 50,
    "top_k_features": 10,
}

config = load_training_config(CONFIG_PATH)
if "n_trials" in config:
    RUN_OPTIONS["n_trials"] = int(config["n_trials"])

summary_df = pd.DataFrame([
    {"item": "day_filter", "value": RUN_OPTIONS["day_filter"]},
    {"item": "day2_flag", "value": RUN_OPTIONS["day2_flag"]},
    {"item": "n_trials", "value": RUN_OPTIONS["n_trials"]},
    {"item": "data_paths", "value": len(config.get("data_paths", []))},
])
summary_df

,item,value
0,day_filter,0
1,day2_flag,False
2,n_trials,20
3,data_paths,2


In [33]:
data_paths = get_data_paths(config)

frames = []
requested_day = int(RUN_OPTIONS["day_filter"])
used_days = []
for p in data_paths:
    p_path = Path(p)
    df_raw = pd.read_csv(p)

    if "day" not in df_raw.columns:
        print(f"skip {p_path.name}: day column not found")
        continue

    df_day = df_raw[df_raw["day"] == requested_day].copy()
    used_day = requested_day

    if df_day.empty:
        available_days = sorted(pd.Series(df_raw["day"]).dropna().unique().tolist())
        if available_days:
            used_day = int(available_days[0])
            df_day = df_raw[df_raw["day"] == used_day].copy()
            print(f"loaded {p_path.name}: raw={len(df_raw)} day={len(df_day)} (fallback day={used_day}, requested={requested_day})")
        else:
            print(f"loaded {p_path.name}: raw={len(df_raw)} day=0 (no valid day values)")
    else:
        print(f"loaded {p_path.name}: raw={len(df_raw)} day={len(df_day)} (day={used_day})")

    if not df_day.empty:
        # 連結元CSVを識別するタグを保持して、2dayと同じ分割・保存構造に合わせる
        df_day["dataset_tag"] = p_path.stem
        used_days.append(used_day)
        frames.append(df_day)

df = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

if df.empty:
    raise ValueError("day フィルタ後のデータが 0 件です。training_config.json の data_paths または day 列を確認してください。")

if used_days:
    RUN_OPTIONS["day_filter"] = int(min(used_days))

label_encoder = LabelEncoder()
df["role_encoded"] = label_encoder.fit_transform(df["role"].astype(str))
y = df["role_encoded"].values

agent_names = df["agent_name"].astype(str).values if "agent_name" in df.columns else np.array(["unknown"] * len(df))

meta = {
    "source_file": df["source_file"].values,
    "dataset_tag": df["dataset_tag"].values if "dataset_tag" in df.columns else np.array(["unknown"] * len(df)),
    "div_result1": df["True_Div_result_1"].values if "True_Div_result_1" in df.columns else np.full(len(df), np.nan),
    "div_id1": df["True_Div_recepient_id_1"].values if "True_Div_recepient_id_1" in df.columns else np.full(len(df), np.nan),
    "div_result2": df["True_Div_result_2"].values if "True_Div_result_2" in df.columns else np.full(len(df), np.nan),
    "div_id2": df["True_Div_recepient_id_2"].values if "True_Div_recepient_id_2" in df.columns else np.full(len(df), np.nan),
    "exec_id": df["exec_id"].values if "exec_id" in df.columns else np.full(len(df), np.nan),
    "attack_id": df["attack_id"].values if "attack_id" in df.columns else np.full(len(df), np.nan),
}

base_drop_cols = [
    "id", "role", "source_file", "dataset_tag", "day", "role_encoded",
    "Est_id_Fact_role", "Est_id_Est_roles", "character_name",
    "agent_name", "combined_text", "seer_co_order"
]
meta_raw_cols = [
    "True_Div_result_1", "True_Div_recepient_id_1",
    "True_Div_result_2", "True_Div_recepient_id_2",
    "exec_id", "attack_id"
]
day2_col = [c for c in df.columns if c.startswith("day1_") or c.startswith("day2_")]
id_like_cols = [c for c in df.columns if c.endswith("_id")]
flag_like_cols = [c for c in df.columns if c.endswith("_flag")]

drop_cols = list(set(base_drop_cols + meta_raw_cols + day2_col + id_like_cols + flag_like_cols))
X_df = df.drop(columns=drop_cols, errors="ignore")
final_features = X_df.columns.tolist()
X = X_df.apply(pd.to_numeric, errors="coerce").fillna(0).values

unique_agents = sorted(pd.Series(agent_names).dropna().astype(str).unique().tolist())
agent_count_by_dataset = (
    df.groupby("dataset_tag")["agent_name"]
    .nunique()
    .reset_index(name="unique_agents")
    .sort_values("dataset_tag")
)

pd.DataFrame([
    {"item": "samples", "value": len(X)},
    {"item": "features", "value": len(final_features)},
    {"item": "roles", "value": ", ".join(label_encoder.classes_)},
    {"item": "games", "value": len(np.unique(meta["source_file"]))},
    {"item": "requested_day", "value": requested_day},
    {"item": "used_days", "value": ", ".join(map(str, sorted(set(used_days)))) if used_days else ""},
    {"item": "dataset_tags", "value": len(np.unique(meta["dataset_tag"]))},
    {"item": "unique_agents", "value": len(unique_agents)},
])

print(f"Unique agents in merged data: {len(unique_agents)}")
print(unique_agents)
print("Unique agents per dataset_tag:")
display(agent_count_by_dataset)

✓ Data files selected:
  - all_feature_table_2025sp17_with_talks.csv
  - all_feature_table_2025summer_with_talks2.csv
loaded all_feature_table_2025sp17_with_talks.csv: raw=1060 day=530 (fallback day=1, requested=0)
loaded all_feature_table_2025summer_with_talks2.csv: raw=1190 day=595 (fallback day=1, requested=0)
Unique agents in merged data: 12
['CamelliaDragons1', 'CanisLupus1', 'Character-Lab1', 'GAIT1', 'GPTaku1', 'ai168wolf1', 'kanolab-nw1', 'kanolab1', 'mille1', 'osawalab1', 'sunamelli1', 'yharada1']
Unique agents per dataset_tag:


,dataset_tag,unique_agents
0,all_feature_table_2025sp17_with_talks,9
1,all_feature_table_2025summer_with_talks2,8


In [35]:
import xgboost as xgb
import optuna
from collections import Counter

print("=" * 70)
print("1DAY START モデル訓練 (Optuna + Balanced Fold Split + Data Leak Prevention)")
print("=" * 70)

t0 = time.time()
predictor = None
run_status = {"ok": False, "error": None, "elapsed_sec": None}

try:
    classes = np.unique(y)
    weights = compute_class_weight(class_weight="balanced", classes=classes, y=y)
    weight_dict = dict(zip(classes, weights))

    groups = np.array([f"{d}::{s}" for d, s in zip(meta["dataset_tag"], meta["source_file"])])
    role_counts = {"POSSESSED": 1, "SEER": 1, "VILLAGER": 2, "WEREWOLF": 1}
    class_names = list(label_encoder.classes_)

    n_trials = int(RUN_OPTIONS["n_trials"])
    n_splits = 5
    fold_seed = 42
    fold_search_iter = 200  # 改善: 80→200で探索強化

    print(f"\n✓ Data prepared for training:")
    print(f"  Samples: {len(X)}")
    print(f"  Features: {len(final_features)}")
    print(f"  Classes: {class_names}")
    print(f"  CV folds: {n_splits}")
    print(f"  Optuna trials: {n_trials}")
    print(f"  Fold search iterations: {fold_search_iter}")

    group_keys = np.array(groups)
    unique_games = np.unique(group_keys)
    dataset_tags = np.array(meta["dataset_tag"])
    source_files = np.array(meta["source_file"])

    game_agent_counter = {}
    game_dataset_counter = {}
    game_pair_counter = {}
    all_agents = set()
    all_datasets = set()
    all_pairs = set()

    for g in unique_games:
        idx = np.where(group_keys == g)[0]
        agent_cnt = Counter(agent_names[i] for i in idx)
        ds_cnt = Counter(dataset_tags[i] for i in idx)
        pair_cnt = Counter((agent_names[i], int(y[i])) for i in idx)

        game_agent_counter[g] = agent_cnt
        game_dataset_counter[g] = ds_cnt
        game_pair_counter[g] = pair_cnt

        all_agents.update(agent_cnt.keys())
        all_datasets.update(ds_cnt.keys())
        all_pairs.update(pair_cnt.keys())

    all_agents = sorted(all_agents)
    all_datasets = sorted(all_datasets)
    all_pairs = sorted(all_pairs)

    def ratio_mse_after_add(fold_counters, add_counter, keys, target_fold):
        total_mse = 0.0
        ideal = 1.0 / n_splits
        eps = 1e-12

        for key in keys:
            vals = []
            for f in range(n_splits):
                base = fold_counters[f].get(key, 0)
                vals.append(base + (add_counter.get(key, 0) if f == target_fold else 0))
            vals = np.array(vals, dtype=float)
            s = vals.sum()
            if s <= eps:
                continue
            ratios = vals / s
            total_mse += float(np.mean((ratios - ideal) ** 2))

        return total_mse / max(len(keys), 1)

    def variance_after_add(fold_counts, add_counter, keys, target_fold):
        total_var = 0.0
        for key in keys:
            vals = []
            for f in range(n_splits):
                base = fold_counts[f].get(key, 0)
                vals.append(base + (add_counter.get(key, 0) if f == target_fold else 0))
            total_var += float(np.var(vals))
        return total_var / max(len(keys), 1)

    def greedy_assign_once(seed: int):
        rng = np.random.default_rng(seed)
        game_order = list(unique_games)
        rng.shuffle(game_order)

        fold_agent_counts = [Counter() for _ in range(n_splits)]
        fold_dataset_counts = [Counter() for _ in range(n_splits)]
        fold_pair_counts = [Counter() for _ in range(n_splits)]
        fold_game_counts = [0 for _ in range(n_splits)]
        game_to_fold = {}

        # 改善: w_pair を 0.35 → 0.45 に上げてagent-role分布を重視
        w_dataset = 1.4
        w_agent = 1.2
        w_pair = 0.45  # agent-role分布をより重視
        w_size = 0.2

        def score_fold_with_candidate(fold_idx, game_id):
            ds_score = ratio_mse_after_add(fold_dataset_counts, game_dataset_counter[game_id], all_datasets, fold_idx)
            agent_score = ratio_mse_after_add(fold_agent_counts, game_agent_counter[game_id], all_agents, fold_idx)
            pair_score = variance_after_add(fold_pair_counts, game_pair_counter[game_id], all_pairs, fold_idx)
            size_vec = np.array(fold_game_counts, dtype=float)
            size_vec[fold_idx] += 1.0
            size_imbalance = float(np.var(size_vec))
            return w_dataset * ds_score + w_agent * agent_score + w_pair * pair_score + w_size * size_imbalance

        for g in game_order:
            best_fold = 0
            best_score = np.inf
            for f in range(n_splits):
                s = score_fold_with_candidate(f, g)
                if s < best_score:
                    best_score = s
                    best_fold = f
            game_to_fold[g] = best_fold
            fold_game_counts[best_fold] += 1
            fold_agent_counts[best_fold].update(game_agent_counter[g])
            fold_dataset_counts[best_fold].update(game_dataset_counter[g])
            fold_pair_counts[best_fold].update(game_pair_counter[g])

        final_score = (
            w_dataset * ratio_mse_after_add(fold_dataset_counts, Counter(), all_datasets, 0)
            + w_agent * ratio_mse_after_add(fold_agent_counts, Counter(), all_agents, 0)
            + w_pair * variance_after_add(fold_pair_counts, Counter(), all_pairs, 0)
            + w_size * float(np.var(np.array(fold_game_counts, dtype=float)))
        )

        return game_to_fold, fold_pair_counts, fold_game_counts, final_score

    best = None
    best_score = np.inf
    scores_history = []
    for k in range(fold_search_iter):
        out = greedy_assign_once(seed=fold_seed + k)
        scores_history.append(out[3])
        if out[3] < best_score:
            best = out
            best_score = out[3]
    
    mean_search_score = float(np.mean(scores_history))
    min_search_score = float(np.min(scores_history))
    print(f"\n✓ Fold search completed:")
    print(f"  Best score: {best_score:.6f}")
    print(f"  Mean score: {mean_search_score:.6f}")
    print(f"  Min score:  {min_search_score:.6f}")

    game_to_fold, fold_pair_counts, fold_game_counts, _ = best
    fold_id_per_row = np.array([game_to_fold[g] for g in group_keys], dtype=int)

    split_indices = []
    for fold in range(n_splits):
        val_idx = np.where(fold_id_per_row == fold)[0]
        train_idx = np.where(fold_id_per_row != fold)[0]
        split_indices.append((train_idx, val_idx))

    fold_summary_df = pd.DataFrame([
        {
            "fold": fold + 1,
            "n_samples": int(np.sum(fold_id_per_row == fold)),
            "n_games": int(len(np.unique(group_keys[fold_id_per_row == fold]))),
        }
        for fold in range(n_splits)
    ])

    fold_role_df = pd.DataFrame([
        {
            "fold": fold + 1,
            "role": label_encoder.inverse_transform([r])[0],
            "count": int(np.sum(y[fold_id_per_row == fold] == r)),
        }
        for fold in range(n_splits)
        for r in np.unique(y)
    ])

    fold_dataset_df = pd.DataFrame([
        {
            "fold": fold + 1,
            "dataset": str(ds),
            "count": int(np.sum(dataset_tags[fold_id_per_row == fold] == ds)),
        }
        for fold in range(n_splits)
        for ds in sorted(np.unique(dataset_tags))
    ])

    fold_agent_df = pd.DataFrame([
        {
            "fold": fold + 1,
            "agent": str(ag),
            "count": int(np.sum(agent_names[fold_id_per_row == fold] == ag)),
        }
        for fold in range(n_splits)
        for ag in sorted(np.unique(agent_names))
    ])

    print("\nBalanced fold summary:")
    display(fold_summary_df)
    print("Fold x Dataset distribution:")
    display(fold_dataset_df.pivot(index="fold", columns="dataset", values="count").fillna(0).astype(int))
    print("Fold x Role distribution:")
    display(fold_role_df.pivot(index="fold", columns="role", values="count").fillna(0).astype(int))
    agent_pivot = fold_agent_df.pivot(index="fold", columns="agent", values="count").fillna(0).astype(int)
    print(f"Fold x Agent distribution (total {len(agent_pivot.columns)} agents):")
    display(agent_pivot)
    
    # 改善: POSSESSED分布の確認
    possessed_dist = fold_role_df[fold_role_df["role"] == "POSSESSED"].set_index("fold")["count"]
    print(f"\n✓ POSSESSED distribution: {dict(possessed_dist)}")
    if possessed_dist.nunique() <= 2:
        print("  ✓ POSSESSED均等配置良好 (各foldに1体配置)")

    models = {}
    cv_results = []

    def evaluate_view_with_constraints(view_name, preds_proba, y_val, meta_val, source_files_val):
        """
        改良版：ゲーム単位での評価（データリークなし）
        各ゲームを個別に評価し、異なるゲーム間のデータ混在を防止
        """
        all_p, all_t = [], []
        eval_games = np.unique(source_files_val)

        for game_id in eval_games:
            game_mask = source_files_val == game_id
            game_indices = np.where(game_mask)[0]
            
            # ゲーム内のサンプル数を確認（5人村なら5）
            if len(game_indices) < 2:
                continue
            
            preds_game = preds_proba[game_indices]
            y_game = y_val[game_indices]
            
            # メタ情報もゲーム単位で抽出
            meta_game = {k: v[game_indices] for k, v in meta_val.items()}
            
            if view_name == "SEER":
                res = assign_roles_for_seer_by_divination(
                    preds_game, y_game, role_counts, class_names, label_encoder,
                    meta_game["div_result1"], meta_game["div_id1"],
                    meta_game["div_result2"], meta_game["div_id2"],
                    meta_game["exec_id"], meta_game["attack_id"],
                    RUN_OPTIONS["day2_flag"],
                )
                for k in ["black", "white"]:
                    if res[k][0].size > 0:
                        all_p.extend(res[k][0])
                        all_t.extend(res[k][1])
            else:
                p, t = assign_roles_for_non_seer(
                    preds_game, y_game, role_counts, class_names, label_encoder,
                    view_name, meta_game["exec_id"], meta_game["attack_id"],
                    RUN_OPTIONS["day2_flag"],
                )
                if p.size > 0:
                    all_p.extend(p)
                    all_t.extend(t)

        if len(all_t) == 0:
            return 0.0, 0

        target_label = "POSSESSED" if view_name == "WEREWOLF" else "WEREWOLF"
        target_id = label_encoder.transform([target_label])[0]
        score = f1_score(all_t, all_p, labels=[target_id], average="macro", zero_division=0)
        return float(score), len(all_t)

    def train_one_view(view_name):
        print(f"\n  {'=' * 65}")
        print(f"  {view_name} 視点モデル: Optuna探索開始")
        print(f"  {'=' * 65}")

        def objective(trial):
            params = {
                "objective": "multi:softprob",
                "num_class": len(classes),
                "eval_metric": "mlogloss",
                "tree_method": "hist",
                "random_state": 42,
                "max_depth": trial.suggest_int("max_depth", 3, 10),
                "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
                "n_estimators": trial.suggest_int("n_estimators", 150, 800),
                "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),
                "subsample": trial.suggest_float("subsample", 0.5, 1.0),
                "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
                "gamma": trial.suggest_float("gamma", 1e-8, 5.0, log=True),
                "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
                "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
            }

            fold_scores = []
            for fold, (train_idx, val_idx) in enumerate(split_indices, start=1):
                X_tr, X_val = X[train_idx], X[val_idx]
                y_tr, y_val = y[train_idx], y[val_idx]
                meta_val = {k: v[val_idx] for k, v in meta.items()}
                source_files_val = source_files[val_idx]
                w_tr = np.array([weight_dict[label] for label in y_tr])

                model = xgb.XGBClassifier(**params)
                model.fit(X_tr, y_tr, sample_weight=w_tr, eval_set=[(X_val, y_val)], verbose=False)
                preds_proba = model.predict_proba(X_val)
                score, _ = evaluate_view_with_constraints(view_name, preds_proba, y_val, meta_val, source_files_val)
                fold_scores.append(score)

                trial.report(np.mean(fold_scores), step=fold)
                if trial.should_prune():
                    raise optuna.TrialPruned()

            return float(np.mean(fold_scores))

        study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
        study.optimize(objective, n_trials=n_trials, show_progress_bar=False)

        best_params = study.best_params.copy()
        best_params.update({
            "objective": "multi:softprob",
            "num_class": len(classes),
            "eval_metric": "mlogloss",
            "tree_method": "hist",
            "random_state": 42,
        })

        print(f"\n    Best trial: {study.best_trial.number}")
        print(f"    Best CV score: {study.best_value:.4f}")
        print(f"    Best params: {study.best_params}")

        fold_scores = []
        fold_samples = []
        for fold, (train_idx, val_idx) in enumerate(split_indices, start=1):
            X_tr, X_val = X[train_idx], X[val_idx]
            y_tr, y_val = y[train_idx], y[val_idx]
            meta_val = {k: v[val_idx] for k, v in meta.items()}
            source_files_val = source_files[val_idx]
            w_tr = np.array([weight_dict[label] for label in y_tr])

            model = xgb.XGBClassifier(**best_params)
            model.fit(X_tr, y_tr, sample_weight=w_tr, eval_set=[(X_val, y_val)], verbose=False)

            preds_proba = model.predict_proba(X_val)
            fold_score, n_assigned = evaluate_view_with_constraints(view_name, preds_proba, y_val, meta_val, source_files_val)
            fold_scores.append(fold_score)
            fold_samples.append(n_assigned)
            print(f"    Fold {fold}/5: assigned={n_assigned}, F1={fold_score:.4f}")

        w_full = np.array([weight_dict[label] for label in y])
        final_model = xgb.XGBClassifier(**best_params)
        final_model.fit(X, y, sample_weight=w_full)
        models[view_name] = final_model

        mean_score = float(np.mean(fold_scores)) if fold_scores else 0.0
        print(f"    ✓ {view_name} モデル訓練完了 (mean F1={mean_score:.4f})")

        cv_results.append({
            "model": view_name,
            "mean_f1": mean_score,
            "best_value": float(study.best_value),
            "fold_scores": fold_scores,
            "assigned_samples": fold_samples,
            "best_trial": int(study.best_trial.number),
            "best_params": study.best_params,
        })

        return final_model

    for view in ["SEER", "WEREWOLF", "VILLAGER", "POSSESSED"]:
        train_one_view(view)

    run_status["ok"] = True
    print(f"\n{'=' * 70}")
    print("✓ 全モデル訓練完了")
    print(f"{'=' * 70}")

    print("\nCV結果サマリー:")
    for result in cv_results:
        print(f"  {result['model']:<10} meanF1={result['mean_f1']:.4f} best_trial={result['best_trial']}")

except Exception as e:
    run_status["error"] = str(e)
    print(f"\nTraining failed: {e}")
    print(traceback.format_exc())
finally:
    run_status["elapsed_sec"] = round(time.time() - t0, 2)

print("\nRun status:")
print(json.dumps(run_status, ensure_ascii=False, indent=2))

1DAY START モデル訓練 (Optuna + Balanced Fold Split + Data Leak Prevention)

✓ Data prepared for training:
  Samples: 1125
  Features: 168
  Classes: ['POSSESSED', 'SEER', 'VILLAGER', 'WEREWOLF']
  CV folds: 5
  Optuna trials: 20
  Fold search iterations: 200

✓ Fold search completed:
  Best score: 0.283534
  Mean score: 0.379574
  Min score:  0.283534

Balanced fold summary:


,fold,n_samples,n_games
0,1,225,45
1,2,225,45
2,3,225,45
3,4,225,45
4,5,225,45


Fold x Dataset distribution:


dataset,all_feature_table_2025sp17_with_talks,all_feature_table_2025summer_with_talks2
fold,,
1,100,125
2,110,115
3,115,110
4,100,125
5,105,120


Fold x Role distribution:


...


    Best trial: 1
    Best CV score: 0.3911
    Best params: {'max_depth': 8, 'learning_rate': 0.010636066512540286, 'n_estimators': 781, 'min_child_weight': 17, 'subsample': 0.6061695553391381, 'colsample_bytree': 0.5909124836035503, 'gamma': 3.939402261362697e-07, 'reg_alpha': 5.472429642032198e-06, 'reg_lambda': 0.00052821153945323}
    Fold 1/5: assigned=360, F1=0.3556
    Fold 2/5: assigned=360, F1=0.4111
    Fold 3/5: assigned=360, F1=0.3556
    Fold 4/5: assigned=360, F1=0.4444
    Fold 5/5: assigned=360, F1=0.3889


[I 2026-04-20 16:18:46,352] A new study created in memory with name: no-name-05b27669-74c9-473a-b451-c3c4959b07c8


    ✓ VILLAGER モデル訓練完了 (mean F1=0.3911)

  POSSESSED 視点モデル: Optuna探索開始


[I 2026-04-20 16:19:05,856] Trial 0 finished with value: 0.32888888888888884 and parameters: {'max_depth': 5, 'learning_rate': 0.17254716573280354, 'n_estimators': 626, 'min_child_weight': 12, 'subsample': 0.5780093202212182, 'colsample_bytree': 0.5779972601681014, 'gamma': 3.200866785899844e-08, 'reg_alpha': 0.6245760287469893, 'reg_lambda': 0.002570603566117598}. Best is trial 0 with value: 0.32888888888888884.
[I 2026-04-20 16:19:34,718] Trial 1 finished with value: 0.3377777777777778 and parameters: {'max_depth': 8, 'learning_rate': 0.010636066512540286, 'n_estimators': 781, 'min_child_weight': 17, 'subsample': 0.6061695553391381, 'colsample_bytree': 0.5909124836035503, 'gamma': 3.939402261362697e-07, 'reg_alpha': 5.472429642032198e-06, 'reg_lambda': 0.00052821153945323}. Best is trial 1 with value: 0.3377777777777778.
[I 2026-04-20 16:19:55,882] Trial 2 finished with value: 0.3333333333333333 and parameters: {'max_depth': 6, 'learning_rate': 0.023927528765580644, 'n_estimators': 5


    Best trial: 3
    Best CV score: 0.3511
    Best params: {'max_depth': 7, 'learning_rate': 0.05898602410432694, 'n_estimators': 180, 'min_child_weight': 13, 'subsample': 0.5852620618436457, 'colsample_bytree': 0.5325257964926398, 'gamma': 1.7960847528705854, 'reg_alpha': 4.905556676028774, 'reg_lambda': 0.18861495878553936}
    Fold 1/5: assigned=180, F1=0.4000
    Fold 2/5: assigned=180, F1=0.2889
    Fold 3/5: assigned=180, F1=0.3556
    Fold 4/5: assigned=180, F1=0.4444
    Fold 5/5: assigned=180, F1=0.2667
    ✓ POSSESSED モデル訓練完了 (mean F1=0.3511)

✓ 全モデル訓練完了

CV結果サマリー:
  SEER       meanF1=0.5511 best_trial=1
  WEREWOLF   meanF1=0.4089 best_trial=7
  VILLAGER   meanF1=0.3911 best_trial=1
  POSSESSED  meanF1=0.3511 best_trial=3

Run status:
{
  "ok": true,
  "error": null,
  "elapsed_sec": 963.37
}


In [7]:
# モデル別の最適パラメータを見やすく表示
if cv_results:
    rows = []
    for r in cv_results:
        row = {"model": r["model"], "best_value": r["best_value"], "best_trial": r["best_trial"]}
        row.update(r["best_params"])
        rows.append(row)
    best_params_df = pd.DataFrame(rows)
    display(best_params_df)
else:
    print("no results")

,model,best_value,best_trial,max_depth,learning_rate,n_estimators,min_child_weight,subsample,colsample_bytree,gamma,reg_alpha,reg_lambda
0,SEER,0.603463,3,7,0.058986,180,13,0.585262,0.532526,1.796085,4.905557,0.188615
1,WEREWOLF,0.486797,44,9,0.010580,332,1,0.750819,0.696334,0.304223,0.000003,0.040970
2,VILLAGER,0.405087,31,10,0.013855,214,18,0.680546,0.801426,0.000176,0.000678,0.000054
3,POSSESSED,0.350000,28,4,0.015612,572,7,0.870396,0.997449,0.004567,0.034634,0.000028


In [11]:
EXPERIMENT_TAG = globals().get("EXPERIMENT_TAG", "baseline")
SPLIT_VERSION = globals().get("SPLIT_VERSION", "dataset_plus_source")

print("=" * 80)
print("Experiment tracking")
print("=" * 80)
print(f"EXPERIMENT_TAG : {EXPERIMENT_TAG}")
print(f"SPLIT_VERSION  : {SPLIT_VERSION}")
print("このタグ名で保存ファイルが分離されます。")

Experiment tracking
EXPERIMENT_TAG : baseline
SPLIT_VERSION  : dataset_plus_source
このタグ名で保存ファイルが分離されます。


In [13]:
# 特徴量重要度（上位k）
feature_rows = []
if models:
    for model_name, model in models.items():
        imp = model.get_booster().get_score(importance_type="weight")
        top_items = sorted(imp.items(), key=lambda x: x[1], reverse=True)[: RUN_OPTIONS["top_k_features"]]
        total = sum(v for _, v in top_items) or 1.0
        for rank, (feat, val) in enumerate(top_items, start=1):
            feature_rows.append(
                {
                    "model": model_name,
                    "rank": rank,
                    "feature": feat,
                    "importance": val,
                    "importance_ratio": val / total,
                }
            )

fi_df = pd.DataFrame(feature_rows)
fi_df.head(20)

,model,rank,feature,importance,importance_ratio
0,SEER,1,f67,399.0,0.278243
1,SEER,2,f15,168.0,0.117155
2,SEER,3,f71,155.0,0.108089
3,SEER,4,f0,125.0,0.087169
4,SEER,5,f31,111.0,0.077406
5,SEER,6,f187,104.0,0.072524
6,SEER,7,f39,100.0,0.069735
7,SEER,8,f79,94.0,0.065551
8,SEER,9,f81,93.0,0.064854
9,SEER,10,f27,85.0,0.059275


In [12]:
import shutil

print("=" * 80)
print("結果の保存")
print("=" * 80)

base_output_dir = PROJECT_ROOT / "notebooks" / "outputs" / "1day_start"
base_output_dir.mkdir(parents=True, exist_ok=True)

exp_tag = globals().get("EXPERIMENT_TAG", "baseline")
split_version = globals().get("SPLIT_VERSION", "dataset_plus_source")
safe_tag = str(exp_tag).replace(" ", "_")
exp_dir = base_output_dir / "experiments" / safe_tag
exp_dir.mkdir(parents=True, exist_ok=True)

def tagged_name(stem, ext):
    return f"{stem}_{safe_tag}.{ext}"

saved_model_name = tagged_name("models_1day_start", "joblib")
saved_model_path = exp_dir / saved_model_name

if models:
    base_model_path = base_output_dir / "models_1day_start.joblib"
    joblib.dump(models, base_model_path)
    joblib.dump(models, saved_model_path)
    print(f"✓ Models saved: {base_model_path}")
    print(f"✓ Tagged models saved: {saved_model_path}")
else:
    print("保存対象のモデルがありません。先に学習セルを実行してください。")

if 'fi_df' in globals() and isinstance(fi_df, pd.DataFrame) and not fi_df.empty:
    base_fi_path = base_output_dir / "feature_importance_1day_start.csv"
    fi_df.to_csv(base_fi_path, index=False)
    fi_tagged_path = exp_dir / tagged_name("feature_importance", "csv")
    fi_df.to_csv(fi_tagged_path, index=False)
    print(f"✓ Feature importance saved: {base_fi_path}")
    print(f"✓ Tagged feature importance saved: {fi_tagged_path}")

if 'fold_id_per_row' in globals():
    fold_assign_df = pd.DataFrame({
        "source_file": meta["source_file"],
        "dataset_tag": meta["dataset_tag"] if "dataset_tag" in meta else "unknown",
        "agent_name": agent_names,
        "role": label_encoder.inverse_transform(y),
        "fold": fold_id_per_row + 1,
    })
    base_fold_assign_path = base_output_dir / "fold_assignments_balanced.csv"
    fold_assign_df.to_csv(base_fold_assign_path, index=False)
    print(f"✓ Fold assignments saved: {base_fold_assign_path}")

    tag_fold_assign_path = exp_dir / tagged_name("fold_assignments_balanced", "csv")
    fold_assign_df.to_csv(tag_fold_assign_path, index=False)
    print(f"✓ Tagged fold assignments saved: {tag_fold_assign_path}")

if 'fold_summary_df' in globals() and isinstance(fold_summary_df, pd.DataFrame):
    base_fold_summary_path = base_output_dir / "fold_split_basic_info.csv"
    fold_summary_df.to_csv(base_fold_summary_path, index=False)
    print(f"✓ Fold split summary saved: {base_fold_summary_path}")

    tag_fold_summary_path = exp_dir / tagged_name("fold_split_basic_info", "csv")
    fold_summary_df.to_csv(tag_fold_summary_path, index=False)
    print(f"✓ Tagged fold split summary saved: {tag_fold_summary_path}")

if 'fold_role_df' in globals() and isinstance(fold_role_df, pd.DataFrame):
    base_fold_role_path = base_output_dir / "fold_role_distribution.csv"
    fold_role_df.to_csv(base_fold_role_path, index=False)
    print(f"✓ Fold role distribution saved: {base_fold_role_path}")

    tag_fold_role_path = exp_dir / tagged_name("fold_role_distribution", "csv")
    fold_role_df.to_csv(tag_fold_role_path, index=False)
    print(f"✓ Tagged fold role distribution saved: {tag_fold_role_path}")

if 'fold_dataset_df' in globals() and isinstance(fold_dataset_df, pd.DataFrame):
    base_fold_dataset_path = base_output_dir / "fold_dataset_distribution.csv"
    fold_dataset_df.to_csv(base_fold_dataset_path, index=False)
    print(f"✓ Fold dataset distribution saved: {base_fold_dataset_path}")

    tag_fold_dataset_path = exp_dir / tagged_name("fold_dataset_distribution", "csv")
    fold_dataset_df.to_csv(tag_fold_dataset_path, index=False)
    print(f"✓ Tagged fold dataset distribution saved: {tag_fold_dataset_path}")

if 'fold_agent_df' in globals() and isinstance(fold_agent_df, pd.DataFrame):
    base_fold_agent_path = base_output_dir / "fold_agent_distribution.csv"
    fold_agent_df.to_csv(base_fold_agent_path, index=False)
    print(f"✓ Fold agent distribution saved: {base_fold_agent_path}")

    tag_fold_agent_path = exp_dir / tagged_name("fold_agent_distribution", "csv")
    fold_agent_df.to_csv(tag_fold_agent_path, index=False)
    print(f"✓ Tagged fold agent distribution saved: {tag_fold_agent_path}")

if 'cv_results' in globals() and isinstance(cv_results, list) and len(cv_results) > 0:
    cv_results_df = pd.DataFrame(cv_results)
    base_cv_json_path = base_output_dir / "cv_results_1day_start.json"
    cv_results_df.to_json(base_cv_json_path, orient="records", force_ascii=False, indent=2)
    print(f"✓ CV results saved: {base_cv_json_path}")

    cv_summary_df = pd.DataFrame([
        {
            "model": r.get("model"),
            "mean_f1": r.get("mean_f1"),
            "best_value": r.get("best_value"),
            "best_trial": r.get("best_trial"),
            "split_version": split_version,
            "experiment_tag": exp_tag,
        }
        for r in cv_results
        if isinstance(r, dict)
    ])
    base_cv_summary_path = base_output_dir / "cv_summary_1day_start.csv"
    cv_summary_df.to_csv(base_cv_summary_path, index=False)
    print(f"✓ CV summary saved: {base_cv_summary_path}")

    tag_cv_path = exp_dir / tagged_name("cv_summary", "csv")
    cv_summary_df.to_csv(tag_cv_path, index=False)
    print(f"✓ Tagged cv_summary saved: {tag_cv_path}")
else:
    print("cv_results がないため CV 出力は保存しません")

meta_out = {
    "generated_at": datetime.now().isoformat(),
    "day_filter": RUN_OPTIONS["day_filter"],
    "day2_flag": RUN_OPTIONS["day2_flag"],
    "n_trials": RUN_OPTIONS["n_trials"],
    "n_samples": int(len(X)),
    "n_features": int(len(final_features)),
    "status": run_status,
    "experiment_tag": exp_tag,
    "split_version": split_version,
    "saved_model_file": saved_model_name,
}
base_meta_path = base_output_dir / "metadata_1day_start.json"
with open(base_meta_path, "w", encoding="utf-8") as f:
    json.dump(meta_out, f, ensure_ascii=False, indent=2)
print(f"✓ Metadata saved: {base_meta_path}")

meta_path = exp_dir / tagged_name("experiment_meta", "json")
with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(meta_out, f, ensure_ascii=False, indent=2)
print(f"✓ Tagged experiment_meta saved: {meta_path}")

print(f"\nSaved directory: {base_output_dir}")
print(f"Tagged directory: {exp_dir}")

結果の保存
✓ Models saved: c:\Users\takic\OneDrive\デスクトップ\修論関係(人狼知能)\役職推定_GitHub\notebooks\outputs\1day_start\models_1day_start.joblib
✓ Tagged models saved: c:\Users\takic\OneDrive\デスクトップ\修論関係(人狼知能)\役職推定_GitHub\notebooks\outputs\1day_start\experiments\baseline\models_1day_start_baseline.joblib
✓ CV results saved: c:\Users\takic\OneDrive\デスクトップ\修論関係(人狼知能)\役職推定_GitHub\notebooks\outputs\1day_start\cv_results_1day_start.json
✓ CV summary saved: c:\Users\takic\OneDrive\デスクトップ\修論関係(人狼知能)\役職推定_GitHub\notebooks\outputs\1day_start\cv_summary_1day_start.csv
✓ Tagged cv_summary saved: c:\Users\takic\OneDrive\デスクトップ\修論関係(人狼知能)\役職推定_GitHub\notebooks\outputs\1day_start\experiments\baseline\cv_summary_baseline.csv
✓ Metadata saved: c:\Users\takic\OneDrive\デスクトップ\修論関係(人狼知能)\役職推定_GitHub\notebooks\outputs\1day_start\metadata_1day_start.json
✓ Tagged experiment_meta saved: c:\Users\takic\OneDrive\デスクトップ\修論関係(人狼知能)\役職推定_GitHub\notebooks\outputs\1day_start\experiments\baseline\experiment_meta_baseline.json

Sa